In [ ]:
import os
import pickle
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam

# === Paths ===
base_dir = "Desktop/photos robot"
model_save_path = "robot_image_model.h5"
label_classes_path = "label_classes.pkl"

# === Settings ===
img_size = (224, 224)
batch_size = 8
epochs = 25
# === Data Generators with Augmentation ===
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    shear_range=0.1,
    horizontal_flip=True,
    validation_split=0.3
)

train_gen = datagen.flow_from_directory(
    base_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='sparse',
    subset='training',
    shuffle=True
)

val_gen = datagen.flow_from_directory(
    base_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='sparse',
    subset='validation',
    shuffle=True
)

# === Save label classes ===
class_indices = train_gen.class_indices
label_classes = [label for label, _ in sorted(class_indices.items(), key=lambda x: x[1])]
with open(label_classes_path, "wb") as f:
    pickle.dump(label_classes, f)

print(f"\n? Classes found: {label_classes}")

# === Fast Class Count ===
print("\n?? Approximate class distribution:")
for class_name in sorted(os.listdir(base_dir)):
    path = os.path.join(base_dir, class_name)
    if os.path.isdir(path):
        count = len(os.listdir(path))
        print(f"{class_name}: {count} images")
        
# === Build the Model ===
base_model = MobileNetV2(input_shape=img_size + (3,), include_top=False, weights='imagenet')
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(len(label_classes), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# === Train the Model ===
print("\n?? Starting base training...")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=epochs
)
model.save(model_save_path)
print(f"\n?? Model saved at: {model_save_path}")
print(f"?? Labels saved at: {label_classes_path}")

# === Fine-tuning ===
print("\n?? Starting fine-tuning...")
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(optimizer=Adam(learning_rate=1e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history_finetune = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10
)

model.save(model_save_path)
print(f"\n? Fine-tuned model saved at: {model_save_path}")        

In [ ]:
import pyttsx3
import time
import pickle
import json
import cv2
import numpy as np
import difflib
from picamera2 import Picamera2
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array

# === Paths ===
MODEL_PATH = "robot_image_model.h5"
LABEL_PATH = "label_classes.pkl"
DATA_PATH = "Downloads/file1.txt"

# === Load model and resources ===
model = load_model(MODEL_PATH)
with open(LABEL_PATH, "rb") as f:
    label_classes = pickle.load(f)
with open(DATA_PATH, "r") as f:
    robot_data = json.load(f)

# === Text-to-speech setup ===
engine = pyttsx3.init()
engine.setProperty("rate", 150)
    print("??", text)
    engine.say(text)
    engine.runAndWait()

# === Robot information lookup ===
def get_robot_info(name):
    names_list = [robot["name"] for robot in robot_data]
    match = difflib.get_close_matches(name, names_list, n=1, cutoff=0.5)
    if match:
        matched_name = match[0]
        for robot in robot_data:
            if robot["name"] == matched_name:
                desc = f"{robot['name']} is made by {robot['manufacturer']}. "
                if "weight_kg" in robot:
                    desc += f"It weighs {robot['weight_kg']} kg. "
                if "payload_kg" in robot:
                    desc += f"It can carry {robot['payload_kg']} kg. "
                if "degrees_of_freedom" in robot:
                    desc += f"It has {robot['degrees_of_freedom']} degrees of freedom. "
                if "power_source" in robot:
                    desc += f"It is powered by {robot['power_source']}. "
                if "functions" in robot:
                    desc += "Functions include " + ", ".join(robot["functions"]) + "."
                return desc
    return "Sorry, I don't have information about this robot."
# === Initialize camera ===
picam2 = Picamera2()
picam2.start()
time.sleep(2)

print("?? Real-time robot recognition running... Press Ctrl+C to stop.")

try:
    last_prediction = ""
    last_time_spoken = 0
    cooldown_seconds = 10  # Wait before repeating the same prediction

    while True:
        frame = picam2.capture_array()
        frame = cv2.cvtColor(frame, cv2.COLOR_RGBA2RGB)
        image = cv2.resize(frame, (224, 224))
        image = img_to_array(image) / 255.0
        image = np.expand_dims(image, axis=0)

        # Predict
        predictions = model.predict(image)[0]
        predicted_index = np.argmax(predictions)
        confidence = predictions[predicted_index]
        robot_name = label_classes[predicted_index]

        current_time = time.time()
        if confidence > 0.7:
            if robot_name != last_prediction or (current_time - last_time_spoken) > cooldown_seconds:
                print(f"? Detected: {robot_name} ({confidence:.2f})")
                speak(f"I see a robot that looks like {robot_name}")
                description = get_robot_info(robot_name)
                speak(description)
                last_prediction = robot_name
                last_time_spoken = current_time

        time.sleep(1)

except KeyboardInterrupt:
    print("?? Stopped by user.")
finally:
    picam2.stop()